# Direct Preference Optimization Walkthrough

Goal: make DPO computable by hand before reading full training code.

You should understand the preference batch `(prompt, chosen, rejected)`, actor/reference log-probability ratios, the pairwise margin, and the role of `beta`.


## Paper-to-code map

Official source:

- `learning/dpo-family/official/repos/direct-preference-optimization/trainers.py`
- `learning/dpo-family/official/repos/direct-preference-optimization/preference_datasets.py`
- `learning/dpo-family/official/repos/direct-preference-optimization/config/loss/dpo.yaml`

Runnable local source:

- `learning/dpo-family/src/dpo_minimal.py`


In [ ]:
from __future__ import annotations
import importlib.util
import math
import platform
from pathlib import Path
import torch
import torch.nn.functional as F
REPO = Path.cwd()
if not (REPO / "learning").exists():
    REPO = Path.cwd().parent
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("repo", REPO)


## Step 1: DPO tensors

DPO reduces each response to four per-sample log-probability sums:

```text
log_p_chosen_actor, log_p_chosen_ref, log_p_rejected_actor, log_p_rejected_ref: (B,)
```


In [ ]:
log_p_chosen_actor = torch.tensor([-8.0, -4.0, -7.5])
log_p_chosen_ref = torch.tensor([-8.5, -4.2, -7.0])
log_p_rejected_actor = torch.tensor([-9.0, -4.7, -7.6])
log_p_rejected_ref = torch.tensor([-8.8, -4.4, -7.1])
chosen_ratio = log_p_chosen_actor - log_p_chosen_ref
rejected_ratio = log_p_rejected_actor - log_p_rejected_ref
margin = chosen_ratio - rejected_ratio
print("chosen_ratio", chosen_ratio)
print("rejected_ratio", rejected_ratio)
print("margin", margin)


## Step 2: loss and beta

If the margin is positive, the actor prefers the chosen answer more than the reference does.


In [ ]:
def dpo_loss_from_logps(a, b, c, d, beta=0.1):
    return -F.logsigmoid(beta * ((a - b) - (c - d))).mean()
for beta in [0.05, 0.1, 0.5, 1.0]:
    print({"beta": beta, "loss": round(float(dpo_loss_from_logps(log_p_chosen_actor, log_p_chosen_ref, log_p_rejected_actor, log_p_rejected_ref, beta)), 4)})


## Step 3: compare with local implementation


In [ ]:
dpo_path = REPO / "learning" / "dpo-family" / "src" / "dpo_minimal.py"
spec = importlib.util.spec_from_file_location("dpo_minimal", dpo_path)
dpo_module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(dpo_module)
local_loss = dpo_module.dpo_loss(log_p_chosen_actor, log_p_chosen_ref, log_p_rejected_actor, log_p_rejected_ref, beta=0.1)
manual_loss = dpo_loss_from_logps(log_p_chosen_actor, log_p_chosen_ref, log_p_rejected_actor, log_p_rejected_ref, beta=0.1)
print(float(local_loss), float(manual_loss))
assert torch.allclose(local_loss, manual_loss)


## Step 4: official source smoke check


In [ ]:
official_root = REPO / "learning" / "dpo-family" / "official" / "repos" / "direct-preference-optimization"
files = [official_root / "trainers.py", official_root / "preference_datasets.py", official_root / "config" / "loss" / "dpo.yaml"]
for file in files:
    print(file.relative_to(REPO), "exists=", file.exists())
assert all(file.exists() for file in files)


## Closed-book checks

1. Explain why `log_p_actor - log_p_ref` behaves like an implicit reward.
2. Draw the shape flow from token logits to scalar DPO loss.
3. Explain the effect of making `beta` too large or too small.
